In [1]:
# Imports
from gliner import GLiNER

import os
import sys
import dotenv
import re

import json
from pathlib import Path

from collections import defaultdict
from prettytable import PrettyTable

import torch
import accelerate

dotenv.load_dotenv()
ROOT_DIR = os.environ.get("ROOT_DIR")
sys.path.append(f"{ROOT_DIR}/scripts")

from evaluation import evaluate

In [2]:
test_before_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_before_2000.json", "r"))
test_after_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_after_2000.json", "r"))

# Raw model

## Inference 

In [3]:
def inference(model, data, tags, threshold):
    all_doc = []
    for doc in data:
        text = doc["texte"]
        entities = []
        # Diviser le texte en chunks si trop long
        max_length = 300  # caractères (pas tokens)
        chunks = []
        start_pos = 0
        while start_pos < len(text):
            # Trouver un point ou une virgule pour couper proprement
            end_pos = min(start_pos + max_length, len(text))
            if end_pos < len(text):
                # Chercher le dernier point/virgule/espace avant la limite
                last_punct = max(
                    text.rfind('. ', start_pos, end_pos),
                    text.rfind('! ', start_pos, end_pos),
                    text.rfind('? ', start_pos, end_pos),
                    text.rfind('\n', start_pos, end_pos)
                )
                if last_punct > start_pos:
                    end_pos = last_punct + 1
            chunk = text[start_pos:end_pos].strip()
            if chunk:
                chunks.append((chunk, start_pos))
            start_pos = end_pos
        
        # Prédire sur chaque chunk
        for chunk_text, chunk_offset in chunks:
            try:
                detected_entities = model.predict_entities(
                    text=chunk_text, 
                    labels=tags, 
                    threshold=threshold
                )
                for ent in detected_entities:
                    # Mapper les labels GLiNER aux tags NER standard
                    label_mapping = {
                        "person": "PER",
                        "location": "LOC",
                        "political party/political movement": "ORG",
                        "profession": "MISC"
                    }
                    
                    original_label = ent["label"].lower()
                    mapped_tag = label_mapping.get(original_label, "MISC")
                    
                    entities.append({
                        "texte": ent["text"],
                        "tag": mapped_tag,
                        "debut": ent["start"] + chunk_offset,
                        "fin": ent["end"] + chunk_offset
                    })
            except Exception as e:
                print(f"Erreur sur chunk {doc['id']}: {e}")
        
        all_doc.append({
            "id": doc["id"],
            "annee": doc["annee"],
            "predicted_entities": entities
        })
    return all_doc

In [4]:
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1") #urchade/gliner_base # urchade/gliner_multi_pii-v1
tags = ["person", "political party/political movement", "location", "profession"]

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]


/Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
for threshold in thresholds:
    print(f"----------------- Running inference with threshold {threshold} -----------------")

    prediction_before_2000 = inference(model, test_before_2000, tags=tags, threshold=threshold)
    prediction_after_2000 = inference(model, test_after_2000, tags=tags, threshold=threshold)

    metrics_before_2000 = evaluate(prediction_before_2000, test_before_2000)
    metrics_after_2000 = evaluate(prediction_after_2000, test_after_2000)

    # Chemin du fichier
    output_path_before_2000 = "../data/results/Gliner/raw/metrics_before_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_before_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_before_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_before_2000, f, indent=2, ensure_ascii=False)

    # Chemin du fichier
    output_path_after_2000 = "../data/results/Gliner/raw/metrics_after_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_after_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_after_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_after_2000, f, indent=2, ensure_ascii=False)

----------------- Running inference with threshold 0.1 -----------------
Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0835 |
|   Recall  | 0.3855 |
|  F1-Score | 0.1373 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1478 |
|   Recall  | 0.6822 |
|  F1-Score | 0.2430 |
+-----------+--------+

Performance by Tag — Exact Match
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0519  | 0.4321 |  0.0926  |    81   |     47     |
| MISC |   0.0767  | 0.3415 |  0.1253  |    82   |     17     |
| ORG  |   0.1070  | 0.2194 |  0.1438  |   196   |     37     |
| PER  |   0.1107  | 0.8551 |  0.1960  |    69   |     26     |
+------+-----------+--------+----------+------

Best threshold = 0,5

# Fine tuned model

In [5]:
def format_data(data, max_length=384, skip_empty=True):
    """
    Format correct pour GLiNER fine-tuning.
    GLiNER attend: tokenized_text (liste de mots) + ner (liste de [start, end, label]).
    Les indices start/end sont en positions de TOKENS (pas de caractères).
    """
    
    formatted_data = []
    
    for doc in data:
        text = doc.get("texte", "").strip()
        entities = doc.get("entites", [])
        
        if not text:
            continue
        
        # Tokenisation simple par mots (compatible GLiNER)
        # On garde track des positions de caractères de chaque token
        token_spans = []  # (token, char_start, char_end)
        for m in re.finditer(r'\S+', text):
            token_spans.append((m.group(), m.start(), m.end()))
        
        if not token_spans:
            continue
        
        # Découper en chunks de max_length tokens (pas caractères)
        chunk_size = max_length
        
        for chunk_start_idx in range(0, len(token_spans), chunk_size):
            chunk_tokens_spans = token_spans[chunk_start_idx:chunk_start_idx + chunk_size]
            
            if not chunk_tokens_spans:
                continue
            
            chunk_tokens = [t[0] for t in chunk_tokens_spans]
            chunk_char_start = chunk_tokens_spans[0][1]
            chunk_char_end = chunk_tokens_spans[-1][2]
            
            # Trouver les entités dans ce chunk
            ner_entities = []
            
            for entity in entities:
                ent_char_start = entity["debut"]
                ent_char_end = entity["fin"]
                # Mapper les tags courts vers les labels utilisés à l'inference
                tag_to_label = {
                    "PER": "person",
                    "LOC": "location",
                    "ORG": "political party/political movement",
                    "MISC": "profession"
                }
                tag = tag_to_label.get(entity["tag"], entity["tag"])
                
                # L'entité doit chevaucher le chunk
                if ent_char_end <= chunk_char_start or ent_char_start >= chunk_char_end:
                    continue
                
                # Trouver le token_start = premier token dont char_start >= ent_char_start
                token_start = None
                token_end = None
                
                for i, (tok, cs, ce) in enumerate(chunk_tokens_spans):
                    # Premier token qui débute dans ou après le début de l'entité
                    if token_start is None and ce > ent_char_start:
                        token_start = i
                    # Dernier token qui finit avant ou à la fin de l'entité
                    if cs < ent_char_end:
                        token_end = i + 1  # exclusif
                
                if token_start is not None and token_end is not None and token_start < token_end:
                    ner_entities.append([token_start, token_end, tag])
            
            # Skip les exemples sans entité si demandé
            if skip_empty and not ner_entities:
                continue
            
            formatted_data.append({
                "tokenized_text": chunk_tokens,
                "ner": ner_entities
            })
    
    return formatted_data

## Training

In [6]:
train_data = json.load(open(f"{ROOT_DIR}/data/splits/train.json", "r"))
train_dataset = format_data(train_data)

In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [8]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1") #urchade/gliner_small

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [9]:
trainer = model.train_model(
    train_dataset=train_dataset,
    eval_dataset=None,
    output_dir="../models/Gliner/",
    learning_rate=1e-4,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    max_steps=1600,             # ← suffit
    save_steps=250,                  # ~1 save par epoch
    save_total_limit=1,
    dataloader_num_workers=0,
    use_cpu=True,
    report_to="none",
    logging_steps=25,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.


Step,Training Loss
25,118.046260
50,27.263728
75,25.017200
100,30.140117
125,19.151605
150,28.156758
175,26.958777
200,29.704866
225,18.009058
250,21.095679


In [11]:
trained_model = GLiNER.from_pretrained(
    os.path.abspath("../models/Gliner/checkpoint-1600"),
    load_tokenizer=True,
    local_files_only=True   # ← forcer la lecture locale
)
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
tags = ["person", "political party/political movement", "location", "profession"]

In [12]:
for threshold in thresholds:
    print(f"----------------- Running inference with threshold {threshold} -----------------")

    prediction_before_2000 = inference(trained_model, test_before_2000, tags=tags, threshold=threshold)
    prediction_after_2000 = inference(trained_model, test_after_2000, tags=tags, threshold=threshold)

    metrics_before_2000 = evaluate(prediction_before_2000, test_before_2000)
    metrics_after_2000 = evaluate(prediction_after_2000, test_after_2000)

    # Chemin du fichier
    output_path_before_2000 = "../data/results/Gliner/trained/trained_metrics_before_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_before_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_before_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_before_2000, f, indent=2, ensure_ascii=False)

    # Chemin du fichier
    output_path_after_2000 = "../data/results/Gliner/trained/trained_metrics_after_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_after_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_after_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_after_2000, f, indent=2, ensure_ascii=False)

----------------- Running inference with threshold 0.1 -----------------
Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0035 |
|   Recall  | 0.0117 |
|  F1-Score | 0.0054 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1947 |
|   Recall  | 0.6519 |
|  F1-Score | 0.2998 |
+-----------+--------+

Performance by Tag — Exact Match
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0000  | 0.0000 |  0.0000  |    81   |     84     |
| MISC |   0.0325  | 0.0488 |  0.0390  |    82   |     13     |
| ORG  |   0.0000  | 0.0000 |  0.0000  |   196   |     85     |
| PER  |   0.0078  | 0.0145 |  0.0101  |    69   |     92     |
+------+-----------+--------+----------+------